# AI-assisted workflow

This notebook repeats the workflow of this week’s lesson, but this time an
AI assistant helps us write the code. The division of labour is simple:

- **You decide the GIS part**: which data, which coordinate reference system,
  which spatial predicate, which classes. You are the GIS expert, and these
  decisions go *into* your prompt.
- **The AI assistant helps with the Python part**: boilerplate, syntax you
  don’t remember, restructuring code, explaining error messages.

The responses shown below are *examples*. If you run the same prompts yourself,
you will get different code, and that is fine. What matters is that you can
read the code, run it, and check the result.

In this first week, we use an AI assistant to write small helper functions
around the *shapely* geometries we created in the lesson.

> **Habit of the week: read before you run**
>
> Read AI-generated code line by line *before* you execute it. Ask yourself:
> what does each line do, what goes in, what comes out? If you see a function
> you don’t know, look it up in the documentation first. Code you have read is
> code you can fix later.

## Your decisions

Before writing any prompt, decide what you actually want:

- Geometries are created from lists of `(x, y)` coordinate tuples (just like in the lesson).
- The geometry type is chosen *explicitly* (`"point"`, `"line"`, `"polygon"`), not guessed from the number of coordinates.
- Invalid input (unknown type, too few coordinates) should raise an error instead of silently producing something.

## Step 1: a function that creates geometries

**Prompt:**

```text
Write a Python function create_geometry(coordinates, geometry_type) using
shapely 2. `coordinates` is a list of (x, y) tuples, `geometry_type` is one of
"point", "line" or "polygon". Raise a ValueError for an unknown geometry type
or if there are too few coordinates for the geometry type. Add a docstring.
```

**Example response** (yours may differ):

In [ ]:
import shapely.geometry


def create_geometry(coordinates, geometry_type):
    """
    Create a shapely geometry from a list of coordinate tuples.

    Parameters
    ----------
    coordinates : list of tuple
        (x, y) coordinate pairs.
    geometry_type : str
        One of "point", "line", or "polygon".

    Returns
    -------
    shapely.geometry.base.BaseGeometry
        The created geometry.
    """
    minimum_coordinates = {"point": 1, "line": 2, "polygon": 3}

    if geometry_type not in minimum_coordinates:
        raise ValueError(f"Unknown geometry type: {geometry_type}")
    if len(coordinates) < minimum_coordinates[geometry_type]:
        raise ValueError(
            f"A {geometry_type} needs at least "
            f"{minimum_coordinates[geometry_type]} coordinate pair(s)"
        )

    if geometry_type == "point":
        return shapely.geometry.Point(coordinates[0])
    if geometry_type == "line":
        return shapely.geometry.LineString(coordinates)
    return shapely.geometry.Polygon(coordinates)

**Before you run it**, read it through:

- The dictionary `minimum_coordinates` encodes the rules from the lesson: a
  line needs at least two coordinate pairs, a polygon at least three.
- For a `"point"`, only the *first* coordinate pair is used. If you pass more,
  the rest is ignored. Is that what you want? For our purposes it is fine, but
  you now *know* it, because you read the code.
- The function returns early (`return` inside the `if`s), so the last line
  only runs for polygons.

Now let’s check that it produces exactly the geometries we created by hand in
the lesson:

In [ ]:
coordinates = [(2.2, 4.2), (7.2, -25.1), (9.26, -2.456)]

point = create_geometry(coordinates[:1], "point")
line = create_geometry(coordinates, "line")
polygon = create_geometry(coordinates, "polygon")

# compare with the geometries created ‘by hand’ in the lesson
assert point.equals(shapely.geometry.Point(2.2, 4.2))
assert line.equals(shapely.geometry.LineString(coordinates))
assert polygon.equals(shapely.geometry.Polygon(coordinates))
assert polygon.is_valid

print(f"Line length: {line.length:.1f} units, polygon area: {polygon.area:.1f} units²")

And that invalid input is, indeed, rejected:

In [ ]:
try:
    create_geometry([(0, 0), (1, 1)], "polygon")
except ValueError as error:
    print(f"As expected: {error}")

## Step 2: points from a data frame, without `iterrows()`

In the hints for exercise 1, we saw three ways to create points from the `x`
and `y` columns of a data frame. Let’s ask for a fourth one:

**Prompt:**

```text
I have a pandas DataFrame `data` with columns "x" and "y". Create a new column
"geometry" that contains shapely Points. Don't use iterrows(). I use shapely 2.
```

**Example response** (yours may differ):

In [ ]:
import pandas
import shapely

data = pandas.DataFrame({"x": [10, 20, 30], "y": [1, 3, 4]})

data["geometry"] = shapely.points(data[["x", "y"]].to_numpy())
data

**Before you run it**: the response uses `shapely.points()`, a function we
have not seen in the lesson. Following this week’s habit, look it up first:
[`shapely.points()` in the shapely
documentation](https://shapely.readthedocs.io/en/stable/reference/shapely.points.html)
creates an array of points from an array of coordinates, all at once
(‘vectorised’), which is much faster than a loop for large data sets.

Does it give the same result as the `apply()` approach from the exercise hints?

In [ ]:
points_with_apply = data.apply(
    lambda row: shapely.geometry.Point(row["x"], row["y"]),
    axis=1
)

assert all(
    point_a.equals(point_b)
    for point_a, point_b in zip(data["geometry"], points_with_apply)
)
print("Both approaches create identical points.")

## AI-use log

Whenever you use an AI assistant in this course (in exercises, only where a
task is marked **AI-LLM OK**, see the [course guidelines on AI
tools](https://autogis-site.readthedocs.io/en/latest/course-info/ai-tools.html)),
keep a short log of what you did. For this notebook, it could look like this:

| Step | What I asked | What I changed | How I checked it |
|------|--------------|----------------|------------------|
| 1 | A function that creates points, lines and polygons from coordinate tuples | Nothing, but noted that points use only the first coordinate pair | Compared with the lesson’s geometries using `assert`, tested invalid input |
| 2 | Points from `x`/`y` columns without `iterrows()` | Nothing | Read the documentation of `shapely.points()`, compared with the `apply()` result |